In [1]:
import pandas as pd
import numpy as np
import os
import json
import difflib

### get the dictionary dimensions and values to verify schema

In [2]:
path='C:/Users/raffi/OneDrive - United Nations/Desktop/DSS/DATA COLLECTOR'
df_translation=pd.read_excel(path+'/translation dict.xlsx')

''' it will create such a structure
{
    "Country": {
        "dim_values": {
            "Egypt": "مصر",
            "Lebanon": "لبنان"
        },
        "dim": {
            "Country": "البلد"
        }
    }
}'''

En_Ar_dictionary={}
#get the unique dimensions
dimensions = set(df_translation['col_en'].unique())

for dim in dimensions:
    df_dim=df_translation[df_translation['col_en'].isin([dim.lower(), dim])].copy()
    En_Ar_dictionary.update(
        {dim:{'dim_values':dict(zip(df_dim['val_en'], df_dim['val_ar'])), 
                'dim': {df_dim['col_en'].unique()[0]:df_dim['col_ar'].unique()[0]}}})

Ar_En_dictionary={}
#get the unique dimensions
dimensions = set(df_translation['col_ar'].unique())

for dim in dimensions:
    df_dim=df_translation[df_translation['col_ar'].isin([dim.lower(), dim])].copy()
    Ar_En_dictionary.update(
        {dim:{'dim_values':dict(zip(df_dim['val_ar'], df_dim['val_en'])), 
                'dim': {df_dim['col_ar'].unique()[0]:df_dim['col_en'].unique()[0]}}})

In [3]:
Ar_En_dictionary.keys()

dict_keys(['أقسام النشاط الإقتصادي', 'سبب الوفاة', 'القطاع المؤسسي', 'الدولة', 'نوع مكان الإقامة', 'الحالة الزوجية', 'مصدر مياه الشرب', 'وضع العمالة', 'المنطقة', 'الفئة', 'نوع حيازة الوحدات السكنية', 'أقسام المهن الرئيسية', 'نوع\xa0الخدمات/المنتجات', 'العدد', 'المصدر', 'المرحلة التعليمية', 'التصنيف الدولي لاسباب الوفاة', 'القطاع', 'المؤشر', 'المواطنة', 'مصدر الإضاءة', 'السنة', 'الفئة العمرية', 'الجنس', 'أسباب البقاء خارج القوى العاملة', 'أنواع نظام التخلص من مياه الصرف الصحي'])

In [4]:
Ar_En_dictionary['الدولة']

{'dim_values': {'الجزائر': 'Algeria',
  'البحرين': 'Bahrain',
  'جزر القمر': 'Comoros Islands',
  'جيبوتي': 'Djibouti',
  'مصر': 'Egypt',
  'العراق': 'Iraq',
  'الأردن': 'Jordan',
  'الكويت': 'Kuwait',
  'لبنان': 'Lebanon',
  'ليبيا': 'Libya',
  'موريتانيا': 'Mauritania',
  'المغرب': 'Morocco',
  'عُمان': 'Oman',
  'فلسطين': 'Palestine',
  'قطر': 'Qatar',
  'السعودية': 'Saudi Arabia',
  'الصومال': 'Somalia',
  'السودان': 'Sudan',
  'الجمهورية العربية السورية': 'Syrian Arab Republic',
  'تونس': 'Tunisia',
  'الإمارات العربية المتحدة': 'United Arab Emirates',
  'اليمن': 'Yemen'},
 'dim': {'الدولة': 'Country'}}

In [5]:
# folder_path='C:/Users/RSHIRINI/OneDrive - United Nations/Desktop/DSS/DATA COLLECTOR/datacollector_received_quest/recieved_quests'

folder_path='C:/Users/raffi/OneDrive - United Nations/Desktop/DSS/DATA COLLECTOR/datacollector_received_quest'
# folder_path='C:/Users/raffi/OneDrive - United Nations/Desktop/DSS/DATA COLLECTOR'
#check all xlsx files in the folder

# Get all excel files in the directory
xlsx_files = [f for f in os.listdir(folder_path) if f.endswith('.xlsx')]

xlsx_files

['Algeria education.xlsx',
 'Algeria health.xlsx',
 'Algeria housing.xlsx',
 'Algeria labor.xlsx',
 'Algeria population.xlsx',
 'Algeria poverty.xlsx',
 'Egypt education.xlsx',
 'Egypt health.xlsx',
 'Egypt housing.xlsx',
 'Egypt labor.xlsx',
 'Egypt population.xlsx',
 'Egypt poverty.xlsx',
 'Iraq health.xlsx',
 'Iraq housing.xlsx',
 'Iraq population.xlsx',
 'Iraq poverty.xlsx',
 'Jordan health.xlsx',
 'Jordan housing.xlsx',
 'Labor Iraq.xlsx',
 'Lebanon poverty.xlsx',
 'Libya education.xlsx',
 'Libya health.xlsx',
 'Libya poverty.xlsx',
 'Mauritania education.xlsx',
 'Mauritania labor.xlsx',
 'Mauritania population.xlsx',
 'Morocco education.xlsx',
 'Morocco health.xlsx',
 'Morocco housing.xlsx',
 'Morocco labor.xlsx',
 'Morocco population.xlsx',
 'Morocco poverty.xlsx',
 'Oman education.xlsx',
 'Oman health.xlsx',
 'Oman housing.xlsx',
 'Oman labor.xlsx',
 'Oman population.xlsx',
 'Qatar education.xlsx',
 'Qatar health.xlsx',
 'Qatar housing.xlsx',
 'Qatar labor.xlsx',
 'Qatar popula

In [12]:
dataframes = []

category_map = {
    'housing': 'سكن',
    'population': 'سكان',
    'labor': 'عمالة',
    'education': 'التعليم',
    'poverty': 'الفقر',
    'health': 'الصحة'
}

#adding الفصل to the dictionary
Ar_En_dictionary['الفصل'] = {'dim_values': {v: k for k, v in category_map.items()}}

for file in xlsx_files:
    print(f'--- Opening Workbook: {file} ---')
    xls = pd.ExcelFile(os.path.join(folder_path, file))
    
    for sheet_name in xls.sheet_names:
        df_raw = pd.read_excel(xls, sheet_name=sheet_name, header=None, dtype=str)
        header_rows = df_raw.index[df_raw[0] == 'index'].tolist()
        if len(header_rows) < 2: continue

        # 1. prepare the data and source slices
        df_data = df_raw[df_raw[0] == '1'].copy()
        df_data.columns = df_raw.iloc[header_rows[0]].dropna().str.strip()
        # Drop index now
        df_data = df_data.drop(columns=['index'], errors='ignore')
        
        df_source = df_raw[df_raw[0] == '2'].copy()
        df_source = df_source[df_raw.iloc[header_rows[1]].dropna().str.strip().index]
        df_source.columns = df_raw.iloc[header_rows[1]].dropna().str.strip()
        # Drop index now
        df_source = df_source.drop(columns=['index'], errors='ignore')

###########################################################################################
        # Extract category and add 'الفصل' column
        def extract_category(sheet_name):
            # Normalize: if there is a '-', take the part after it, otherwise use the whole name
            target = sheet_name.split('-')[-1].strip()
    
            # Now extract the part before the underscore (if it exists)
            return target.split('_')[0].strip().lower()

        raw_cat = extract_category(sheet_name)

        # If raw_cat is not in keys, try to find the best match using difflib
        if raw_cat not in category_map.keys():
            matches = difflib.get_close_matches(raw_cat, category_map.keys(), n=1, cutoff=0.6)
            if matches:
                best_match = matches[0]
                print(f"  >> FIX CATEGORY: {raw_cat} -> {best_match}")
                raw_cat = best_match
            else:
                print(f'  !!! ERROR: Could not identify category for {raw_cat} from {file}')

        # Assign the value based on the (potentially fixed) raw_cat
        df_data['الفصل'] = category_map.get(raw_cat, raw_cat)

########################################################################################################
        
        # --- 3. merge data with source ---
        # id_vars are all non-digit columns (Indicator, Country, Chapter, etc.)
        id_vars = [c for c in df_data.columns if not str(c).isdigit()]
        # year_columns are only the digit columns (2010, 2011, etc.)
        year_columns = [c for c in df_data.columns if str(c).isdigit()]
        
        #DEBUG############
        # print(sheet_name)
        #DEBUG############

        df_melted = df_data.melt(
            id_vars=id_vars, 
            value_vars=year_columns, 
            var_name='السنة', 
            value_name='العدد'
        )

        # CRASH LOCATOR BLOCK
        try:
            # Merge melted data with source info
            df_merged = pd.merge(df_melted, df_source, on=['السنة', 'المؤشر', 'الدولة'], how='left')
            
            
        except KeyError as e:
            print("\n" + "="*50)
            print(f"💥 CRASH DETECTED IN WORKBOOK: {file}")
            print(f"💥 ON SHEET: {sheet_name}")
            print(f"💥 FOR CHAPTER ('الفصل'): {df_data['الفصل'].iloc[0] if 'الفصل' in df_data.columns else 'Unknown'}")
            print("="*50)
            print(f"Columns actually found in df_data:\n{list(df_data.columns)}\n")
            print(f"Columns actually found in df_source:\n{list(df_source.columns)}")
            print("="*50)
            raise e
        
###############################################################################################################        
        # --- CLEANING BLOCK ---

        # Added 'الفصل' to keep it from being changed/fixed
        standard_cols = {'السنة', 'العدد', 'المصدر', 'الفصل'}
        
        # 1. Column Name Cleanup
        search_targets = list(standard_cols) + list(Ar_En_dictionary.keys())
        
        for col in list(df_merged.columns):
            match = difflib.get_close_matches(str(col), search_targets, n=1, cutoff=0.6)
            if match and match[0] != col:
                best_match = match[0]
                print(f"  >> FIX COL: {sheet_name} : {col}({len(str(col))} chars) -> {best_match} ({len(str(best_match))} chars)")
                df_merged.rename(columns={col: best_match}, inplace=True)
                
        # 2. Cell Value Cleanup
        for col in df_merged.columns:
            # Note: Ensuring column exists in dict to avoid KeyError
            if col in Ar_En_dictionary:
                allowed_vals = [str(k) for k in Ar_En_dictionary[col]['dim_values'].keys()]
                
                for val in df_merged[col].unique():
                    if str(val) not in allowed_vals:
                        v_match = difflib.get_close_matches(str(val), allowed_vals, n=1, cutoff=0.6)
                        if v_match and v_match[0] != str(val):
                            print(f"FIX VAL: [{sheet_name} : {col}] {val} ({len(str(val))} chars) -> {v_match[0]} ({len(str(v_match[0]))} chars)")
                            df_merged[col] = df_merged[col].replace(val, v_match[0])
            else:
                print(f'{col} was not found in the dictionary')

        #append to the dataframes
        dataframes.append(df_merged)
                             

########################################################################################################

--- Opening Workbook: Algeria education.xlsx ---
FIX VAL: [Education_4 : الجنس]  إناث (5 chars) -> إناث (4 chars)
FIX VAL: [Education_3 : الجنس]  إناث (5 chars) -> إناث (4 chars)
FIX VAL: [Education_2_c : الجنس]  إناث (5 chars) -> إناث (4 chars)
FIX VAL: [Education_2_b : الجنس]  إناث (5 chars) -> إناث (4 chars)
FIX VAL: [Education_2_a : الجنس]  إناث (5 chars) -> إناث (4 chars)
FIX VAL: [Education_1_c : الجنس]  إناث (5 chars) -> إناث (4 chars)
FIX VAL: [Education_1_b : الجنس]  إناث (5 chars) -> إناث (4 chars)
FIX VAL: [Education_1_a : الجنس]  إناث (5 chars) -> إناث (4 chars)
--- Opening Workbook: Algeria health.xlsx ---
FIX VAL: [Health_12 : الفئة العمرية] 15-24 (5 chars) -> 15-24 سنة (9 chars)
FIX VAL: [Health_11_b : الفئة العمرية] 15-24 (5 chars) -> 15-24 سنة (9 chars)
FIX VAL: [Health_11_a : الفئة العمرية] 15-24 (5 chars) -> 15-24 سنة (9 chars)
FIX VAL: [Health_10 : الفئة العمرية] 15-24 (5 chars) -> 15-24 سنة (9 chars)
FIX VAL: [Health_2_a : المؤشر] الولادات التي يشرف عليها أخصائيون 

In [ ]:
# Final aggregation
path='C:/Users/raffi/OneDrive - United Nations/Desktop/DSS/COMPENDIUM-ARAB SOCIETY'
if dataframes:
    final_df = pd.concat(dataframes, ignore_index=True)
    
    # One last safety strip in case different sheets had different trailing spaces 
    # that survived the merge or concat
    final_df.columns = final_df.columns.str.strip()
    
    final_df = final_df.dropna(axis=1, how='all')
    final_df.to_excel(path+'/final_dataset_combined.xlsx', index=False)
    print("\nSuccess: Combined data from all workbooks and sheets.")
else:
    print("\nNo data found. Check your folder path and sheet structures.")


Success: Combined data from all workbooks and sheets.


In [10]:
final_df.columns

Index(['المؤشر', 'الدولة', 'الفصل', 'السنة', 'العدد', 'المصدر', 'المواطنة',
       'الجنس', 'المرحلة التعليمية', 'الفئة العمرية', 'المنطقة',
       'مصدر الإضاءة', 'أنواع نظام التخلص من مياه الصرف الصحي',
       'نوع حيازة الوحدات السكنية', 'مصدر مياه الشرب', 'نوع مكان الإقامة',
       'القطاع المؤسسي', 'أقسام المهن الرئيسية', 'أقسام النشاط الإقتصادي',
       'وضع العمالة', 'أسباب البقاء خارج القوى العاملة',
       'التصنيف الدولي لاسباب الوفاة', 'سبب الوفاة', 'الحالة الزوجية', 'الفئة',
       'نوع الخدمات/المنتجات', 'القطاع'],
      dtype='object')

### check all the column names if they are  unique

In [9]:
import difflib

cols = list(final_df.columns)
threshold = 0.9

for col in cols:
    # Find everything in the list that is "close enough" to the current column
    # By setting it to len(cols), you are telling Python: "Don't stop at 3 (default),
    # show me every single match you find in the entire list."
    matches = difflib.get_close_matches(col, cols, n=len(cols), cutoff=threshold)
    
    # If it found more than itself, you have a duplicate issue
    if len(matches) > 1:
        print(f"Similarity found for '{col}': {matches}")

### check unique values in columns

In [16]:
import difflib

# Keep THRESHOLD very high to only catch typos/spaces, not different indicators
THRESHOLD = 0.98 
fuzzy_report = {}

# Compare each value with every other value in the list
'''enumerate(values): This gives you two things at once: the index i (the position) and the val1 (the actual text, like "الجزائر").
This loop picks the "Anchor" item. We are going to hold this item in our hand and compare it to others.'''
 
for column, values in Ar_En_dictionary.items():
    similar_pairs = []
    
    # FIX: Convert 'values' to a list so Python knows how to slice it properly
    values_list = list(values)

    for i, val1 in enumerate(values_list):
        for val2 in values_list[i+1:]:
            # Use raw strings to catch hidden spaces
            s1, s2 = str(val1), str(val2)
            score = difflib.SequenceMatcher(None, s1, s2).ratio()
            
            #Only show if they are ALMOST identical but NOT exactly 1.0
            if score >= THRESHOLD and score < 1.0:
                # Get the sources
                src1 = final_df[final_df[column] == val1]['file_source'].unique().tolist()
                src2 = final_df[final_df[column] == val2]['file_source'].unique().tolist()
                
                # Format exactly as you requested
                # Wrapping values in quotes "" helps you see the trailing spaces in the terminal
                report_entry = f"\"{val1}\" {src1} <<<--->>> \"{val2}\" {src2}"
                similar_pairs.append(report_entry)
                
    if similar_pairs:
        fuzzy_report[column] = similar_pairs

# Print the diagnostic report
print(json.dumps(fuzzy_report, indent=4, ensure_ascii=False))


{}
